# Smart MCQ Solver — From-Scratch TF-IDF + Residual Neural Network

**Model type :** Built entirely from scratch (no pretrained transformers)  
**Features  :** 90+ hand-engineered features per question (TF-IDF, BM25, Jaccard, lexical, syntactic)  

In [1]:
!pip install numpy pandas scikit-learn torch rank-bm25 tqdm wandb

## 1. Install & Import Dependencies

In [2]:
import wandb

wandb.login(key="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")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [3]:
import os
import re
import string
import warnings
import math
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from tqdm import tqdm


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device  : {DEVICE}")
print(f"Device Name : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"PyTorch : {torch.__version__}")

OPTIONS = ["A", "B", "C", "D", "E"]


Device  : cuda
Device Name : Tesla T4
PyTorch : 2.10.0+cu128


## 2. Data Loading & EDA

In [4]:
if os.path.exists("/kaggle"):
    TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
    TEST_PATH  = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
else:
    TRAIN_PATH = "../Data/train.csv"
    TEST_PATH  = "../Data/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("=" * 55)
print(f"Train : {train_df.shape}   Test : {test_df.shape}")
print("=" * 55)
print(train_df.info())
print()
print("Answer Distribution:")
print(train_df["answer"].value_counts())


Train : (2000, 8)   Test : (500, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB
None

Answer Distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


## 3. Rich Text Preprocessing

Unlike the basic version, we now:
- **Preserve negation** ("not" becomes "NEG_not") so the model can tell "does X" from "does NOT X"
- **Preserve numbers and units** — removing them loses arithmetic information
- **Apply lemmatization** — maps inflected forms to their base (running -> run)
- **Remove only unhelpful punctuation** (keep `?`, `%`, `+`, `-`, `=`)

In [5]:
NEGATION_WORDS = {
    "not", "no", "never", "neither", "nor", "none", "nothing",
    "nowhere", "nobody", "cannot", "can't", "don't", "doesn't",
    "didn't", "isn't", "aren't", "wasn't", "weren't", "won't",
    "wouldn't", "shouldn't", "couldn't", "except", "unless"
}

KEEP_PUNCT = set("?%+-=<>/_")
REMOVE_PUNCT = "".join(c for c in string.punctuation if c not in KEEP_PUNCT)


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower()

    tokens = text.split()
    marked = ["NEG_" + t if t.rstrip(string.punctuation) in NEGATION_WORDS else t
              for t in tokens]
    text = " ".join(marked)

    translator = str.maketrans("", "", REMOVE_PUNCT)
    text = text.translate(translator)

    return " ".join(text.split())


def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["prompt_clean"] = df["prompt"].apply(clean_text)
    for opt in OPTIONS:
        df[f"{opt}_clean"] = df[opt].apply(clean_text)
    return df


print("Preprocessing Train...")
train_df = preprocess_df(train_df)
print("Preprocessing Test...")
test_df  = preprocess_df(test_df)

sample = "Which of the following is NOT a property of water?"
print(f"\nOriginal : {sample}")
print(f"Cleaned  : {clean_text(sample)}")


Preprocessing Train...
Preprocessing Test...

Original : Which of the following is NOT a property of water?
Cleaned  : which of the following is NEG_not a property of water?


## 4. TF-IDF Vectorizer (Shared Vocabulary)

In [6]:
fit_corpus = pd.concat(
    [train_df["prompt_clean"]] + [train_df[f"{o}_clean"] for o in OPTIONS],
    ignore_index=True
)

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=10_000,
    sublinear_tf=True,
    min_df=2,
)
tfidf.fit(fit_corpus)

print(f"TF-IDF Vocabulary : {len(tfidf.vocabulary_):,} Terms")
print(f"Corpus Size : {len(fit_corpus):,} Documents")


TF-IDF Vocabulary : 10,000 Terms
Corpus Size : 12,000 Documents


## 5. Rich Feature Engineering — 90+ Features Per Question

For **every option** we compute 15 features, giving 75 option-level features.  
We also add 15 global question-level features. Total: ~90 features.

| Category | Features per option |
|----------|--------------------|
| Similarity | TF-IDF cosine, BM25, Jaccard, char-trigram overlap |
| Lexical | Common word count, word overlap ratio, char overlap |
| Length | Option length, length ratio, absolute length diff |
| Syntactic | Contains negation, number, unit, comparison words |

| Category | Global features |
|----------|----------------|
| Question type | is_arithmetic, is_science, is_geography, is_coding |
| Special tokens | contains NOT/EXCEPT/TRUE/FALSE, contains year, equation |
| Prompt stats | prompt length, unique words, avg option length |

In [7]:
def tokenize(text: str) -> list:
    return text.lower().split()


def jaccard_similarity(a: str, b: str) -> float:
    sa, sb = set(tokenize(a)), set(tokenize(b))
    if not sa and not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def common_word_count(a: str, b: str) -> int:
    return len(set(tokenize(a)) & set(tokenize(b)))


def word_overlap_ratio(prompt: str, option: str) -> float:
    p, o = set(tokenize(prompt)), set(tokenize(option))
    if not p:
        return 0.0
    return len(p & o) / len(p)


def char_ngram_overlap(a: str, b: str, n: int = 3) -> float:
    ga = set(a[i:i+n] for i in range(len(a)-n+1))
    gb = set(b[i:i+n] for i in range(len(b)-n+1))
    if not ga and not gb:
        return 0.0
    return len(ga & gb) / len(ga | gb)


def length_ratio(prompt: str, option: str) -> float:
    return len(tokenize(option)) / (len(tokenize(prompt)) + 1)


def contains_negation(text: str) -> int:
    return int("NEG_" in text)


def contains_number(text: str) -> int:
    return int(bool(re.search(r"\d", text)))


UNITS = [
    "kg", "km", "cm", "mm", "meter", "metre", "gram", "liter", "litre",
    "mol", "kelvin", "joule", "watt", "volt", "ampere", "hz", "mhz",
    "ghz", "byte", "kb", "mb", "gb", "%", "percent", "celsius", "fahrenheit"
]


def contains_unit(text: str) -> int:
    text_lower = text.lower()
    return int(any(u in text_lower for u in UNITS))


COMPARISON_WORDS = [
    "more", "less", "greater", "smaller", "larger", "fewer",
    "most", "least", "best", "worst", "highest", "lowest",
    "maximum", "minimum", "increase", "decrease"
]


def contains_comparison(text: str) -> int:
    text_lower = text.lower()
    return int(any(w in text_lower.split() for w in COMPARISON_WORDS))


def bm25_score(prompt_tokens: list, option_tokens: list) -> float:
    if not option_tokens:
        return 0.0
    bm25 = BM25Okapi([option_tokens])
    scores = bm25.get_scores(prompt_tokens)
    return float(scores[0])


print("Helper Functions Defined.")


Helper Functions Defined.


In [8]:
QUESTION_TYPE_KEYWORDS = {
    "is_arithmetic": ["calculate", "compute", "how many", "sum", "difference",
                       "product", "divide", "total", "average", "mean", "ratio"],
    "is_science": ["element", "atom", "molecule", "chemical", "reaction",
                    "physics", "biology", "force", "energy", "cell", "dna"],
    "is_geography": ["country", "capital", "continent", "river", "mountain",
                     "ocean", "city", "nation", "border", "location"],
    "is_coding": ["algorithm", "function", "variable", "loop", "class",
                   "python", "java", "code", "programming", "runtime"],
    "is_history": ["war", "century", "dynasty", "revolution", "ancient",
                    "founded", "battle", "emperor", "colonist", "civilization"],
}


def global_question_features(row: pd.Series) -> dict:
    prompt  = row["prompt_clean"]
    p_lower = prompt.lower()

    feats = {}

    for qtype, keywords in QUESTION_TYPE_KEYWORDS.items():
        feats[qtype] = int(any(kw in p_lower for kw in keywords))

    feats["has_not"]      = int("NEG_" in prompt)
    feats["has_except"]   = int("NEG_except" in prompt)
    feats["has_true"]     = int(" true " in f" {p_lower} ")
    feats["has_false"]    = int(" false " in f" {p_lower} ")
    feats["has_year"]     = int(bool(re.search(r"\b(1[0-9]{3}|20[0-9]{2})\b", prompt)))
    feats["has_equation"] = int(any(c in prompt for c in ["=", "+", "-", "/"]))

    p_tokens = tokenize(prompt)
    feats["prompt_len"]     = len(p_tokens)
    feats["prompt_unique"]  = len(set(p_tokens))

    opt_lens = [len(tokenize(row[f"{o}_clean"])) for o in OPTIONS]
    feats["avg_opt_len"]  = float(np.mean(opt_lens))
    feats["std_opt_len"]  = float(np.std(opt_lens))
    feats["max_opt_len"]  = float(np.max(opt_lens))
    feats["min_opt_len"]  = float(np.min(opt_lens))

    return feats


print("Global Feature Extractor Defined.")


Global Feature Extractor Defined.


In [9]:
def option_features(row: pd.Series, vectorizer: TfidfVectorizer) -> dict:
    prompt = row["prompt_clean"]
    p_tok  = tokenize(prompt)
    feats  = {}

    prompt_vec = vectorizer.transform([prompt])
    prompt_has_neg = contains_negation(prompt)
    prompt_has_num = contains_number(prompt)

    for opt in OPTIONS:
        option = row[f"{opt}_clean"]
        o_tok  = tokenize(option)

        opt_vec = vectorizer.transform([option])
        cos_sim = cosine_similarity(prompt_vec, opt_vec)[0, 0]

        bm25 = bm25_score(p_tok, o_tok)

        jacc = jaccard_similarity(prompt, option)

        char3 = char_ngram_overlap(prompt, option, n=3)

        cw    = common_word_count(prompt, option)
        norm_cw = cw / (len(set(p_tok)) + 1)

        wo = word_overlap_ratio(prompt, option)

        lr = length_ratio(prompt, option)

        ld = abs(len(o_tok) - len(p_tok)) / (len(p_tok) + 1)

        ol = len(o_tok)

        neg  = contains_negation(option)
        num  = contains_number(option)
        unit = contains_unit(option)
        comp = contains_comparison(option)

        neg_xor = int(prompt_has_neg != neg)

        num_match = int(prompt_has_num and num)

        feats[f"{opt}_tfidf_cosine"]   = float(cos_sim)
        feats[f"{opt}_bm25"]           = float(bm25)
        feats[f"{opt}_jaccard"]        = float(jacc)
        feats[f"{opt}_char_trigram"]   = float(char3)
        feats[f"{opt}_common_words"]   = float(norm_cw)
        feats[f"{opt}_word_overlap"]   = float(wo)
        feats[f"{opt}_len_ratio"]      = float(lr)
        feats[f"{opt}_len_diff"]       = float(ld)
        feats[f"{opt}_opt_len"]        = float(ol)
        feats[f"{opt}_has_negation"]   = float(neg)
        feats[f"{opt}_has_number"]     = float(num)
        feats[f"{opt}_has_unit"]       = float(unit)
        feats[f"{opt}_has_comparison"] = float(comp)
        feats[f"{opt}_neg_xor"]        = float(neg_xor)
        feats[f"{opt}_num_match"]      = float(num_match)

    return feats


print("Per-Option Feature Extractor Defined.")
print(f"Features Per Question: 5 Options X 15 = 75 Option Features + ~16 Global = ~91 Total")


Per-Option Feature Extractor Defined.
Features Per Question: 5 Options X 15 = 75 Option Features + ~16 Global = ~91 Total


In [10]:
def build_feature_matrix(df: pd.DataFrame, vectorizer: TfidfVectorizer) -> np.ndarray:
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Building features"):
        g = global_question_features(row)
        o = option_features(row, vectorizer)
        combined = {**g, **o}
        rows.append(combined)

    feat_df = pd.DataFrame(rows)
    return feat_df.values.astype(np.float32), feat_df.columns.tolist()


print("Building Train Feature Matrix...")
X_all, feature_names = build_feature_matrix(train_df, tfidf)
print(f"Train Feature Matrix : {X_all.shape}")

print("Building Test Feature Matrix...")
X_test, _ = build_feature_matrix(test_df, tfidf)
print(f"Test Feature Matrix : {X_test.shape}")


Building Train Feature Matrix...


Building features: 100%|██████████| 2000/2000 [00:15<00:00, 127.03it/s]


Train Feature Matrix : (2000, 92)
Building Test Feature Matrix...


Building features: 100%|██████████| 500/500 [00:04<00:00, 124.16it/s]

Test Feature Matrix : (500, 92)


In [11]:
scaler = StandardScaler()
X_all_scaled  = scaler.fit_transform(X_all)
X_test_scaled = scaler.transform(X_test)

label_encoder = LabelEncoder()
label_encoder.fit(OPTIONS)
y_all = label_encoder.transform(train_df["answer"])

NUM_FEATURES = X_all_scaled.shape[1]
print(f"Scaled Features : {X_all_scaled.shape}")
print(f"Feature Names   : {feature_names[:5]} ... ({NUM_FEATURES} Total)")


Scaled Features : (2000, 92)
Feature Names   : ['is_arithmetic', 'is_science', 'is_geography', 'is_coding', 'is_history'] ... (92 Total)


## 6. MAP@3 Metric

In [12]:
def map_at_k(actual: list, predicted: list, k: int = 3) -> float:
    scores = []
    for act, preds in zip(actual, predicted):
        ap = 0.0
        for rank, pred in enumerate(preds[:k], start=1):
            if pred == act:
                ap = 1.0 / rank
                break
        scores.append(ap)
    return float(np.mean(scores))


def get_top3_preds(logits: np.ndarray) -> list:
    ranked = np.argsort(logits, axis=1)[:, ::-1]
    return [[OPTIONS[i] for i in row[:3]] for row in ranked]


assert abs(map_at_k(["A"], [["A","B","C"]]) - 1.0  ) < 1e-9
assert abs(map_at_k(["A"], [["B","A","C"]]) - 0.5  ) < 1e-9
assert abs(map_at_k(["A"], [["C","D","A"]]) - 1/3  ) < 1e-9
assert abs(map_at_k(["A"], [["B","C","D"]]) - 0.0  ) < 1e-9
print("MAP@3 Sanity Checks Passed")


MAP@3 Sanity Checks Passed


## 7. Residual MLP Architecture

Standard deep networks can suffer from vanishing gradients and degradation.  
**Residual connections** (`x + F(x)`) allow gradients to flow directly and let
the network learn *what to change* rather than the whole mapping from scratch.

In [13]:
class ResidualBlock(nn.Module):

    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(out_dim, out_dim),
            nn.BatchNorm1d(out_dim),
        )
        self.skip = (
            nn.Linear(in_dim, out_dim, bias=False)
            if in_dim != out_dim else nn.Identity()
        )
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.block(x) + self.skip(x))


class MCQResidualNet(nn.Module):

    def __init__(
        self,
        num_features: int,
        hidden_dim: int    = 512,
        dropout: float     = 0.3,
        num_res_blocks: int = 3,
    ):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Linear(num_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout),
        )

        dims = [hidden_dim]
        for i in range(1, num_res_blocks + 1):
            dims.append(max(hidden_dim // (2 ** i), 32))

        self.res_blocks = nn.ModuleList([
            ResidualBlock(dims[i], dims[i+1], dropout=dropout)
            for i in range(num_res_blocks)
        ])

        self.head = nn.Linear(dims[-1], 5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        for block in self.res_blocks:
            x = block(x)
        return self.head(x)


_test_model = MCQResidualNet(num_features=NUM_FEATURES, hidden_dim=512)
_dummy = torch.zeros(4, NUM_FEATURES)
assert _test_model(_dummy).shape == (4, 5), "Shape mismatch!"
total_params = sum(p.numel() for p in _test_model.parameters() if p.requires_grad)
print(_test_model)
print(f"\nTotal Trainable Parameters: {total_params:,}")


MCQResidualNet(
  (stem): Sequential(
    (0): Linear(in_features=92, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
  )
  (res_blocks): ModuleList(
    (0): ResidualBlock(
      (block): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): GELU(approximate='none')
        (3): Dropout(p=0.3, inplace=False)
        (4): Linear(in_features=256, out_features=256, bias=True)
        (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (skip): Linear(in_features=512, out_features=256, bias=False)
      (act): GELU(approximate='none')
    )
    (1): ResidualBlock(
      (block): Sequential(
        (0): Linear(in_features=256, out_features=128, bias=True)
        (1): Batc

## 8. Training Utilities

- **Label Smoothing** - CrossEntropyLoss(label_smoothing=0.1) prevents overconfidence
- **Early Stopping** - halts training when validation MAP@3 stops improving
- **OneCycleLR** - warm-up + cosine decay scheduler

In [14]:
class EarlyStopping:

    def __init__(self, patience: int = 10, min_delta: float = 1e-4, mode: str = "max"):
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self.counter    = 0
        self.best_score = None
        self.stop       = False
        self.best_state = None

    def __call__(self, score: float, model: nn.Module) -> bool:
        improved = (
            self.best_score is None or
            (self.mode == "max" and score > self.best_score + self.min_delta) or
            (self.mode == "min" and score < self.best_score - self.min_delta)
        )
        if improved:
            self.best_score = score
            self.counter    = 0
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


def make_dataloader(
    X: np.ndarray,
    y: np.ndarray | None = None,
    batch_size: int = 128,
    shuffle: bool = True
) -> DataLoader:
    X_t = torch.tensor(X, dtype=torch.float32)
    if y is not None:
        dataset = TensorDataset(X_t, torch.tensor(y, dtype=torch.long))
    else:
        dataset = TensorDataset(X_t)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def evaluate_model(model: nn.Module, loader: DataLoader, criterion) -> dict:
    model.eval()
    all_logits, all_labels = [], []
    total_loss, n = 0.0, 0

    with torch.no_grad():
        for batch in loader:
            X_b, y_b = batch[0].to(DEVICE), batch[1].to(DEVICE)
            logits = model(X_b)
            total_loss += criterion(logits, y_b).item()
            n += 1
            all_logits.append(logits.cpu().numpy())
            all_labels.append(y_b.cpu().numpy())

    all_logits = np.vstack(all_logits)
    all_labels = np.concatenate(all_labels)

    preds_top1   = np.argmax(all_logits, axis=1)
    accuracy     = float(np.mean(preds_top1 == all_labels))
    preds_top3   = get_top3_preds(all_logits)
    actual_strs  = [OPTIONS[i] for i in all_labels]
    map3         = map_at_k(actual_strs, preds_top3)

    return {"accuracy": accuracy, "map3": map3, "loss": total_loss / n}


print("Training Utilities Defined.")


Training Utilities Defined.


## 9. Hyperparameter Configuration

Sensible defaults are used instead of automated HPO to keep things simple.

In [15]:
FINAL_CONFIG = {
    "model"          : "TF-IDF + Residual MLP (From Scratch)",
    "num_features"   : NUM_FEATURES,
    "hidden_dim"     : 256,
    "dropout"        : 0.3,
    "lr"             : 5e-4,
    "weight_decay"   : 1e-4,
    "batch_size"     : 128,
    "label_smoothing": 0.1,
    "num_res_blocks" : 2,
    "epochs"         : 80,
    "patience"       : 10,
    "n_folds"        : 5,
    "optimizer"      : "AdamW",
    "scheduler"      : "OneCycleLR",
    "seed"           : SEED,
}
print("Config:")
for k, v in FINAL_CONFIG.items():
    print(f"  {k:<20} : {v}")


Config:
  model                : TF-IDF + Residual MLP (From Scratch)
  num_features         : 92
  hidden_dim           : 256
  dropout              : 0.3
  lr                   : 0.0005
  weight_decay         : 0.0001
  batch_size           : 128
  label_smoothing      : 0.1
  num_res_blocks       : 2
  epochs               : 80
  patience             : 10
  n_folds              : 5
  optimizer            : AdamW
  scheduler            : OneCycleLR
  seed                 : 42


## 10. Stratified 5-Fold Cross Validation with W&B Tracking

Stratified K-Fold ensures each fold has the same answer distribution as the full
dataset. We train one model per fold and average their test probabilities for a
simple ensemble.

In [16]:
WANDB_PROJECT = "24F2008474-t22026"

skf = StratifiedKFold(n_splits=FINAL_CONFIG["n_folds"], shuffle=True, random_state=SEED)

oof_logits   = np.zeros((len(X_all_scaled), 5), dtype=np.float32)
test_logits  = np.zeros((len(X_test_scaled), 5), dtype=np.float32)

fold_map3_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_all_scaled, y_all), start=1):

    print(f"\n{'='*55}")
    print(f" FOLD {fold}/{FINAL_CONFIG['n_folds']}")
    print(f"{'='*55}")

    run = wandb.init(
        project=WANDB_PROJECT,
        name=f"tfidf-residual-nn-fold{fold}",
        config={**FINAL_CONFIG, "fold": fold},
        tags=["from-scratch", "tfidf", "residual-mlp", "kfold"],
        reinit=True,
    )

    X_tr, X_val = X_all_scaled[tr_idx], X_all_scaled[val_idx]
    y_tr, y_val = y_all[tr_idx],         y_all[val_idx]

    BS = FINAL_CONFIG["batch_size"]
    tr_loader  = make_dataloader(X_tr,  y_tr,  batch_size=BS, shuffle=True)
    val_loader = make_dataloader(X_val, y_val, batch_size=BS, shuffle=False)

    model = MCQResidualNet(
        num_features  = NUM_FEATURES,
        hidden_dim    = FINAL_CONFIG["hidden_dim"],
        dropout       = FINAL_CONFIG["dropout"],
        num_res_blocks= FINAL_CONFIG["num_res_blocks"],
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=FINAL_CONFIG["label_smoothing"])
    optimizer = optim.AdamW(
        model.parameters(),
        lr=FINAL_CONFIG["lr"],
        weight_decay=FINAL_CONFIG["weight_decay"]
    )
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=FINAL_CONFIG["lr"],
        steps_per_epoch=len(tr_loader),
        epochs=FINAL_CONFIG["epochs"],
    )

    early_stop = EarlyStopping(
        patience=FINAL_CONFIG["patience"], min_delta=1e-4, mode="max"
    )

    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>8} | {'Val Acc':>7} | {'Val MAP@3':>9}")
    print("-" * 55)

    for epoch in range(1, FINAL_CONFIG["epochs"] + 1):
        model.train()
        train_loss, n_b = 0.0, 0
        for X_b, y_b in tr_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
            n_b += 1

        avg_train_loss = train_loss / n_b
        val_metrics    = evaluate_model(model, val_loader, criterion)
        current_lr     = optimizer.param_groups[0]["lr"]

        wandb.log({
            "epoch"        : epoch,
            "train/loss"   : avg_train_loss,
            "val/loss"     : val_metrics["loss"],
            "val/accuracy" : val_metrics["accuracy"],
            "val/map3"     : val_metrics["map3"],
            "learning_rate": current_lr,
        })

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"{epoch:>6} | {avg_train_loss:>10.4f} | "
                f"{val_metrics['loss']:>8.4f} | "
                f"{val_metrics['accuracy']:>7.4f} | "
                f"{val_metrics['map3']:>9.4f}"
            )

        if early_stop(val_metrics["map3"], model):
            print(f"  Early Stopping At Epoch {epoch} (patience={FINAL_CONFIG['patience']})")
            break

    model.load_state_dict(early_stop.best_state)
    fold_map3_scores.append(early_stop.best_score)

    wandb.summary["best_val_map3"]     = early_stop.best_score
    wandb.summary["best_val_accuracy"] = val_metrics["accuracy"]

    print(f"  Fold {fold} Best MAP@3 : {early_stop.best_score:.4f}")

    model.eval()
    oof_logits_fold = []
    with torch.no_grad():
        for batch in val_loader:
            oof_logits_fold.append(
                model(batch[0].to(DEVICE)).cpu().numpy()
            )
    oof_logits[val_idx] = np.vstack(oof_logits_fold)

    test_loader = make_dataloader(X_test_scaled, batch_size=BS, shuffle=False)
    test_logits_fold = []
    with torch.no_grad():
        for batch in test_loader:
            test_logits_fold.append(
                model(batch[0].to(DEVICE)).cpu().numpy()
            )
    test_logits += np.vstack(test_logits_fold) / FINAL_CONFIG["n_folds"]

    wandb.finish()


print("\n" + "=" * 55)
print(f"  K-Fold Results")
print("=" * 55)
for i, s in enumerate(fold_map3_scores, 1):
    print(f"  Fold {i} MAP@3 : {s:.4f}")
print(f"  Mean  MAP@3 : {np.mean(fold_map3_scores):.4f} +/- {np.std(fold_map3_scores):.4f}")



 FOLD 1/5


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260726_175821-0z1wi6jx
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tfidf-residual-nn-fold1
wandb: ⭐️ View project at https://wandb.ai/spreadsheets600-student/24F2008474-t22026
wandb: 🚀 View run at https://wandb.ai/spreadsheets600-student/24F2008474-t22026/runs/0z1wi6jx


 Epoch | Train Loss | Val Loss | Val Acc | Val MAP@3
-------------------------------------------------------
     1 |     1.6646 |   1.6190 |  0.2000 |    0.3617
    10 |     1.4452 |   1.3747 |  0.5600 |    0.7212
    20 |     0.7866 |   0.6578 |  0.8800 |    0.9325
    30 |     0.5846 |   0.5209 |  0.9375 |    0.9654
    40 |     0.5081 |   0.4646 |  0.9775 |    0.9883
    50 |     0.4771 |   0.4444 |  0.9775 |    0.9888


wandb: updating run metadata


  Early Stopping At Epoch 53 (patience=10)
  Fold 1 Best MAP@3 : 0.9921


wandb: uploading history steps 0-52, summary, console lines 0-9
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
wandb: learning_rate ▁▁▁▁▂▃▃▄▅▅▆▆▇▇▇██████████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄
wandb:    train/loss ████▇▇▇▇▆▆▅▅▄▄▄▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:  val/accuracy ▁▁▂▃▃▄▄▅▅▅▆▆▇▇▇▇▇▇▇▇████████████████████
wandb:      val/loss █████▇▇▆▆▅▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val/map3 ▁▁▁▂▂▄▄▅▅▅▆▆▆▇▇▇▇▇█▇████████████████████
wandb: 
wandb: Run summary:
wandb: best_val_accuracy 0.98
wandb:     best_val_map3 0.99208
wandb:             epoch 53
wandb:     learning_rate 0.00023
wandb:        train/loss 0.47003
wandb:      val/accuracy 0.98
wandb:          val/loss 0.44078
wandb:          val/map3 0.99
wandb: 
wandb: 🚀 View run tfidf-residual-nn-fold1 at: https://wandb.ai/spreadsheets600-student/24F2008474-t22026/runs/0z1wi6jx
wandb: ⭐️ View project at: https://wandb.ai/spreadsheets600-student/24F2008474-t22026
wandb: Synced 5 W&B file(s), 0 media file(s),

... (17 outputs left out)

wandb: setting up run tvvhqu98
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260726_175853-tvvhqu98
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tfidf-residual-nn-fold5
wandb: ⭐️ View project at https://wandb.ai/spreadsheets600-student/24F2008474-t22026
wandb: 🚀 View run at https://wandb.ai/spreadsheets600-student/24F2008474-t22026/runs/tvvhqu98


 Epoch | Train Loss | Val Loss | Val Acc | Val MAP@3
-------------------------------------------------------
     1 |     1.6742 |   1.6140 |  0.2350 |    0.3887
    10 |     1.4556 |   1.3648 |  0.5725 |    0.7262
    20 |     0.8181 |   0.7031 |  0.8975 |    0.9342
    30 |     0.6036 |   0.4856 |  0.9750 |    0.9862
    40 |     0.5043 |   0.4505 |  0.9825 |    0.9904
    50 |     0.4772 |   0.4401 |  0.9850 |    0.9925
    60 |     0.4576 |   0.4305 |  0.9950 |    0.9975
    70 |     0.4605 |   0.4288 |  0.9925 |    0.9962
  Early Stopping At Epoch 71 (patience=10)
  Fold 5 Best MAP@3 : 0.9988


wandb: updating run metadata
wandb: uploading history steps 0-70, summary, console lines 1-11
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇████
wandb: learning_rate ▁▁▁▂▂▃▄▅▆▆▇███████▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▁
wandb:    train/loss ██▇▇▇▆▅▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:  val/accuracy ▁▁▁▂▃▄▅▆▆▇▇▇████████████████████████████
wandb:      val/loss ████▇▆▆▅▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val/map3 ▁▁▃▄▄▅▅▆▆▆▇▇████████████████████████████
wandb: 
wandb: Run summary:
wandb: best_val_accuracy 0.99
wandb:     best_val_map3 0.99875
wandb:             epoch 71
wandb:     learning_rate 3e-05
wandb:        train/loss 0.45619
wandb:      val/accuracy 0.99
wandb:          val/loss 0.42721
wandb:          val/map3 0.995
wandb: 
wandb: 🚀 View run tfidf-residual-nn-fold5 at: https://wandb.ai/spreadsheets600-student/24F2008474-t22026/runs/tvvhqu98
wandb: ⭐️ View project at: https://wandb.ai/spreadsheets600-student/24F2008474-t22026
wandb: Synced 5 


  K-Fold Results
  Fold 1 MAP@3 : 0.9921
  Fold 2 MAP@3 : 0.9946
  Fold 3 MAP@3 : 0.9938
  Fold 4 MAP@3 : 0.9975
  Fold 5 MAP@3 : 0.9988
  Mean  MAP@3 : 0.9953 +/- 0.0024


## 11. OOF (Out-of-Fold) Evaluation

In [17]:
oof_preds_top3 = get_top3_preds(oof_logits)
oof_map3       = map_at_k(train_df["answer"].tolist(), oof_preds_top3)

oof_top1       = np.argmax(oof_logits, axis=1)
oof_accuracy   = float(np.mean(oof_top1 == y_all))

print("=" * 45)
print(" OUT-OF-FOLD PERFORMANCE (Full Train Set)")
print("=" * 45)
print(f"  Accuracy : {oof_accuracy:.4f}")
print(f"  MAP@3    : {oof_map3:.4f}")


 OUT-OF-FOLD PERFORMANCE (Full Train Set)
  Accuracy : 0.9910
  MAP@3    : 0.9953


## 12. Error Analysis

In [18]:
error_df = train_df[["prompt", "answer"]].copy()
error_df["top1_pred"]    = [OPTIONS[i] for i in oof_top1]
error_df["top3_preds"]   = [",".join(p) for p in oof_preds_top3]
error_df["top1_correct"] = error_df["answer"] == error_df["top1_pred"]
error_df["in_top3"]      = [
    ans in preds
    for ans, preds in zip(error_df["answer"], oof_preds_top3)
]

print(f"Top-1 Accuracy : {error_df['top1_correct'].mean():.4f}")
print(f"Top-3 Accuracy : {error_df['in_top3'].mean():.4f}")
print()

conf = pd.crosstab(
    error_df["answer"],
    error_df["top1_pred"],
    rownames=["Actual"],
    colnames=["Predicted"]
)
print("Confusion Matrix (Top-1):")
print(conf)

"\nSample Misclassified Questions:"
miss = error_df[~error_df["top1_correct"]]
print(miss[["prompt", "answer", "top1_pred", "top3_preds"]].head(5).to_string(index=False))


Top-1 Accuracy : 0.9910
Top-3 Accuracy : 1.0000

Confusion Matrix (Top-1):
Predicted    A    B    C    D    E
Actual                            
A          368    1    0    0    0
B            1  488    0    1    0
C            1    5  450    2    1
D            0    0    1  354    3
E            1    0    1    0  322
                                                                                                                                                                                        prompt answer top1_pred top3_preds
Which of the following is correct? What are the two main interpretations for the disparity between the presence of matter and antimatter in the observable universe? among the listed options.      D         C      C,E,D
                                                                                                                                    What is the effect generated by a spinning superconductor?      C         B      B,C,D
                       

## 13. Generate Test Predictions & Kaggle Submission

In [19]:
test_preds_top3 = get_top3_preds(test_logits)

submission_df = pd.DataFrame({
    "id"        : test_df["id"],
    "prediction": [" ".join(p) for p in test_preds_top3],
})

print(f"Submission Shape : {submission_df.shape}")
print(submission_df.head(10).to_string(index=False))


Submission Shape : (500, 2)
 id prediction
  1      A E B
  2      B C D
  3      B E A
  4      E A B
  5      C E A
  6      D C B
  7      E D B
  8      B E C
  9      C B D
 10      B C A


In [20]:
if os.path.exists("/kaggle"):
    SUBMISSION_PATH = "submission.csv"
else:
    SUBMISSION_PATH = "../Submissions/submission_tfidf_residual_nn.csv"

submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission Saved : {SUBMISSION_PATH}")

final_run = wandb.init(
    project=WANDB_PROJECT,
    name="tfidf-residual-nn-final-submission",
    config=FINAL_CONFIG,
    tags=["from-scratch", "submission"],
    reinit=True,
)
wandb.summary["oof_map3"]     = oof_map3
wandb.summary["oof_accuracy"] = oof_accuracy
wandb.summary["mean_fold_map3"] = float(np.mean(fold_map3_scores))

artifact = wandb.Artifact("submission-tfidf-residual-nn", type="submission")
artifact.add_file(SUBMISSION_PATH)
wandb.log_artifact(artifact)
wandb.finish()

print("Submission Artifact Logged To W&B")


Submission Saved : submission.csv


wandb: setting up run w3ptok0z
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260726_175901-w3ptok0z
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tfidf-residual-nn-final-submission
wandb: ⭐️ View project at https://wandb.ai/spreadsheets600-student/24F2008474-t22026
wandb: 🚀 View run at https://wandb.ai/spreadsheets600-student/24F2008474-t22026/runs/w3ptok0z
wandb: updating run metadata; uploading artifact submission-tfidf-residual-nn
wandb: uploading artifact submission-tfidf-residual-nn
wandb: 
wandb: Run summary:
wandb: mean_fold_map3 0.99533
wandb:   oof_accuracy 0.991
wandb:       oof_map3 0.99533
wandb: 
wandb: 🚀 View run tfidf-residual-nn-final-submission at: https://wandb.ai/spreadsheets600-student/24F2008474-t22026/runs/w3ptok0z
wandb: ⭐️ View project at: https://wandb.ai/spreadsheets600-student/24F2008474-t22026
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(

Submission Artifact Logged To W&B


## 14. Save Model Weights & Finish

In [21]:
if os.path.exists("/kaggle"):
    MODEL_PATH = "tfidf_residual_nn_model.pt"
else:
    MODEL_PATH = "../Models/tfidf_residual_nn_model.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": FINAL_CONFIG,
        "best_val_map3": np.mean(fold_map3_scores),
    },
    MODEL_PATH,
)
print(f"Model Checkpoint Saved : {MODEL_PATH}")

model_artifact = wandb.Artifact("tfidf-residual-nn-model", type="model")
model_artifact.add_file(MODEL_PATH)

wandb.init(project=WANDB_PROJECT, reinit=True)
wandb.log_artifact(model_artifact)
wandb.finish()
print("\nW&B Run Finished")


Model Checkpoint Saved : tfidf_residual_nn_model.pt


wandb: setting up run 6wvq57eo
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260726_175904-6wvq57eo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run fearless-blaze-27
wandb: ⭐️ View project at https://wandb.ai/spreadsheets600-student/24F2008474-t22026
wandb: 🚀 View run at https://wandb.ai/spreadsheets600-student/24F2008474-t22026/runs/6wvq57eo
wandb: updating run metadata; uploading artifact tfidf-residual-nn-model; uploading summary
wandb: uploading artifact tfidf-residual-nn-model
wandb: uploading data
wandb: 🚀 View run fearless-blaze-27 at: https://wandb.ai/spreadsheets600-student/24F2008474-t22026/runs/6wvq57eo
wandb: ⭐️ View project at: https://wandb.ai/spreadsheets600-student/24F2008474-t22026
wandb: Synced 4 W&B file(s), 0 media file(s), 2 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260726_175904-6wvq57eo/logs



W&B Run Finished


---
## Summary

| Component | Details |
|-----------|--------------------------------------------------|
| **Model** | Residual Feed-Forward MLP (built from scratch) |
| **Features** | ~91: TF-IDF cosine . BM25 . Jaccard . char-trigram . word overlap . length ratio . negation . number . unit . comparison . question-type indicators |
| **Text Cleaning** | Lemmatization . negation marking . unit/number preservation |
| **TF-IDF** | Unigram+Bigram . max 10K features . sublinear TF |
| **Architecture** | Stem -> 3x ResidualBlock(BN+GELU+Dropout) -> head |
| **Loss** | CrossEntropyLoss with label smoothing |
| **Optimizer** | AdamW + gradient clipping |
| **Scheduler** | OneCycleLR (warmup + cosine decay) |
| **Regularisation** | BatchNorm . Dropout . Weight decay . Early Stopping |
| **Validation** | Stratified 5-Fold CV + OOF MAP@3 |
| **HPO** | Manual (Grid-Free Defaults) |
| **Tracking** | Weights & Biases |
| **Output** | `submission_tfidf_residual_nn.csv` -- ensemble of 5 folds |